In [1]:
import ee
import json
import pandas as pd
from pathlib import Path

# --------------------------------------------------
# Earth Engine
# --------------------------------------------------

ee.Initialize(project="development-hruday")


# --------------------------------------------------
# Paths
# --------------------------------------------------

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

BOUNDARY_FILE = (
    PROJECT_ROOT
    / "raw"
    / "boundary_file"
    / "380006_boundary.geojson"
)

CLIMATE_DIR = (
    PROJECT_ROOT
    / "raw"
    / "climate"
)

CLIMATE_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# --------------------------------------------------
# Load pincode boundary
# --------------------------------------------------

with open(BOUNDARY_FILE, "r") as f:
    geojson_data = json.load(f)

coords = geojson_data["features"][0]["geometry"]["coordinates"]

roi = ee.Geometry.Polygon(coords)


print("Earth Engine initialized.")
print("Boundary loaded.")
print("Climate directory:", CLIMATE_DIR)

Earth Engine initialized.
Boundary loaded.
Climate directory: c:\dev\case-study\raw\climate


In [2]:
REPRESENTATIVE_LON = 72.60115569230976
REPRESENTATIVE_LAT = 22.999693294401865

representative_point = ee.Geometry.Point([
    REPRESENTATIVE_LON,
    REPRESENTATIVE_LAT
])

print("Representative ERA5-Land pixel:")
print("Longitude:", REPRESENTATIVE_LON)
print("Latitude :", REPRESENTATIVE_LAT)

Representative ERA5-Land pixel:
Longitude: 72.60115569230976
Latitude : 22.999693294401865


In [3]:
VARIABLES = {
    "solar_radiation": "surface_solar_radiation_downwards_sum",
    "temperature": "temperature_2m",
    "precipitation": "total_precipitation_sum"
}

In [4]:
CONVERSIONS = {
    "solar_radiation": 1 / 3_600_000,  # J/m² → kWh/m²
    "temperature": None,                # K → °C handled separately
    "precipitation": 1000               # m → mm
}

In [5]:
def extract_year(year, band_name):

    start_date = f"{year}-01-01"
    end_date = f"{year + 1}-01-01"

    collection = (
        ee.ImageCollection("ECMWF/ERA5_LAND/DAILY_AGGR")
        .filterDate(start_date, end_date)
        .select(band_name)
    )

    def extract_image(image):

        value = image.reduceRegion(
            reducer=ee.Reducer.first(),
            geometry=representative_point,
            scale=11132
        ).get(band_name)

        return ee.Feature(
            None,
            {
                "date": image.date().format("YYYY-MM-dd"),
                "latitude": REPRESENTATIVE_LAT,
                "longitude": REPRESENTATIVE_LON,
                "value": value
            }
        )

    features = collection.map(extract_image)

    # This year's collection is only ~365 records,
    # so it stays safely below the 5000-element limit.
    info = features.getInfo()

    rows = [
        feature["properties"]
        for feature in info["features"]
    ]

    return pd.DataFrame(rows)

In [6]:
df_test = extract_year(
    2005,
    VARIABLES["solar_radiation"]
)

print(df_test.shape)

display(df_test.head())
display(df_test.tail())

(365, 4)


,date,latitude,longitude,value
0,2005-01-01,22.999693,72.601156,16986888.0
1,2005-01-02,22.999693,72.601156,17057332.0
2,2005-01-03,22.999693,72.601156,17375624.0
3,2005-01-04,22.999693,72.601156,17006828.0
4,2005-01-05,22.999693,72.601156,16805996.0


,date,latitude,longitude,value
360,2005-12-27,22.999693,72.601156,17068332.0
361,2005-12-28,22.999693,72.601156,16997160.0
362,2005-12-29,22.999693,72.601156,16680208.0
363,2005-12-30,22.999693,72.601156,16738224.0
364,2005-12-31,22.999693,72.601156,16947940.0


In [7]:
all_ssrd = []

for year in range(2005, 2025):

    print(f"Downloading SSRD: {year}...")

    df_year = extract_year(
        year,
        VARIABLES["solar_radiation"]
    )

    all_ssrd.append(df_year)

    print(
        f"  → {len(df_year)} observations"
    )

df_ssrd = pd.concat(
    all_ssrd,
    ignore_index=True
)

df_ssrd["date"] = pd.to_datetime(
    df_ssrd["date"]
)

df_ssrd = df_ssrd.sort_values(
    "date"
).reset_index(drop=True)

print()
print("Total observations:", len(df_ssrd))
print(
    "Date range:",
    df_ssrd["date"].min(),
    "→",
    df_ssrd["date"].max()
)

  → 365 observations
  → 365 observations
  → 365 observations
  → 366 observations
  → 365 observations
  → 365 observations
  → 365 observations
  → 366 observations
  → 365 observations
  → 365 observations
  → 365 observations
  → 366 observations
  → 365 observations
  → 365 observations
  → 365 observations
  → 366 observations
  → 365 observations
  → 365 observations
  → 365 observations
  → 366 observations

Total observations: 7305
Date range: 2005-01-01 00:00:00 → 2024-12-31 00:00:00


In [8]:
df_ssrd["ssrd_kwh_m2"] = (
    df_ssrd["value"] / 3_600_000
)

In [9]:
df_ssrd = df_ssrd.rename(
    columns={
        "value": "ssrd_j_m2"
    }
)

In [10]:
ssrd_file = (
    CLIMATE_DIR
    / "ERA5_SSRD_380006_2005_2024.csv"
)

df_ssrd.to_csv(
    ssrd_file,
    index=False
)

print("Saved:")
print(ssrd_file)

Saved:
c:\dev\case-study\raw\climate\ERA5_SSRD_380006_2005_2024.csv


In [11]:
all_temperature = []

for year in range(2005, 2025):

    print(f"Downloading temperature: {year}...")

    df_year = extract_year(
        year,
        VARIABLES["temperature"]
    )

    all_temperature.append(df_year)

df_temperature = pd.concat(
    all_temperature,
    ignore_index=True
)

df_temperature["date"] = pd.to_datetime(
    df_temperature["date"]
)

df_temperature = df_temperature.rename(
    columns={
        "value": "temperature_k"
    }
)

# Kelvin → Celsius
df_temperature["temperature_c"] = (
    df_temperature["temperature_k"] - 273.15
)

df_temperature = df_temperature.sort_values(
    "date"
).reset_index(drop=True)

temperature_file = (
    CLIMATE_DIR
    / "ERA5_Temperature_380006_2005_2024.csv"
)

df_temperature.to_csv(
    temperature_file,
    index=False
)

print("Saved:")
print(temperature_file)

Saved:
c:\dev\case-study\raw\climate\ERA5_Temperature_380006_2005_2024.csv


In [12]:
all_precipitation = []

for year in range(2005, 2025):

    print(f"Downloading precipitation: {year}...")

    df_year = extract_year(
        year,
        VARIABLES["precipitation"]
    )

    all_precipitation.append(df_year)

df_precipitation = pd.concat(
    all_precipitation,
    ignore_index=True
)

df_precipitation["date"] = pd.to_datetime(
    df_precipitation["date"]
)

df_precipitation = df_precipitation.rename(
    columns={
        "value": "precipitation_m"
    }
)

# metres → millimetres
df_precipitation["precipitation_mm"] = (
    df_precipitation["precipitation_m"] * 1000
)

df_precipitation = df_precipitation.sort_values(
    "date"
).reset_index(drop=True)

precipitation_file = (
    CLIMATE_DIR
    / "ERA5_Precipitation_380006_2005_2024.csv"
)

df_precipitation.to_csv(
    precipitation_file,
    index=False
)

print("Saved:")
print(precipitation_file)

Saved:
c:\dev\case-study\raw\climate\ERA5_Precipitation_380006_2005_2024.csv


In [13]:
import pandas as pd

ssrd = pd.read_csv(
    CLIMATE_DIR / "ERA5_SSRD_380006_2005_2024.csv"
)

temperature = pd.read_csv(
    CLIMATE_DIR / "ERA5_temperature_380006_2005_2024.csv"
)

precipitation = pd.read_csv(
    CLIMATE_DIR / "ERA5_precipitation_380006_2005_2024.csv"
)

print("SSRD:")
print(ssrd.shape)

print("\nTemperature:")
print(temperature.shape)

print("\nPrecipitation:")
print(precipitation.shape)

SSRD:
(7305, 5)

Temperature:
(7305, 5)

Precipitation:
(7305, 5)


In [14]:
display(ssrd.head())
display(temperature.head())
display(precipitation.head())

,date,latitude,longitude,ssrd_j_m2,ssrd_kwh_m2
0,2005-01-01,22.999693,72.601156,16986888.0,4.718580
1,2005-01-02,22.999693,72.601156,17057332.0,4.738148
2,2005-01-03,22.999693,72.601156,17375624.0,4.826562
3,2005-01-04,22.999693,72.601156,17006828.0,4.724119
4,2005-01-05,22.999693,72.601156,16805996.0,4.668332


,date,latitude,longitude,temperature_k,temperature_c
0,2005-01-01,22.999693,72.601156,290.894920,17.744920
1,2005-01-02,22.999693,72.601156,291.209627,18.059627
2,2005-01-03,22.999693,72.601156,291.418463,18.268463
3,2005-01-04,22.999693,72.601156,292.641347,19.491347
4,2005-01-05,22.999693,72.601156,293.143656,19.993656


,date,latitude,longitude,precipitation_m,precipitation_mm
0,2005-01-01,22.999693,72.601156,8.404254e-07,0.000840
1,2005-01-02,22.999693,72.601156,8.660680e-07,0.000866
2,2005-01-03,22.999693,72.601156,8.583069e-07,0.000858
3,2005-01-04,22.999693,72.601156,8.553266e-07,0.000855
4,2005-01-05,22.999693,72.601156,8.611054e-07,0.000861


In [15]:
for name, df in {
    "SSRD": ssrd,
    "Temperature": temperature,
    "Precipitation": precipitation
}.items():

    df["date"] = pd.to_datetime(df["date"])

    print(
        f"{name}: "
        f"{df['date'].min().date()} → "
        f"{df['date'].max().date()}"
    )

SSRD: 2005-01-01 → 2024-12-31
Temperature: 2005-01-01 → 2024-12-31
Precipitation: 2005-01-01 → 2024-12-31


In [16]:
for name, df in {
    "SSRD": ssrd,
    "Temperature": temperature,
    "Precipitation": precipitation
}.items():

    print(f"\n{name}")
    print(df.isna().sum())


SSRD
date           0
latitude       0
longitude      0
ssrd_j_m2      0
ssrd_kwh_m2    0
dtype: int64

Temperature
date             0
latitude         0
longitude        0
temperature_k    0
temperature_c    0
dtype: int64

Precipitation
date                0
latitude            0
longitude           0
precipitation_m     0
precipitation_mm    0
dtype: int64


In [17]:
print(ssrd.columns)
print(temperature.columns)
print(precipitation.columns)

Index(['date', 'latitude', 'longitude', 'ssrd_j_m2', 'ssrd_kwh_m2'], dtype='str')
Index(['date', 'latitude', 'longitude', 'temperature_k', 'temperature_c'], dtype='str')
Index(['date', 'latitude', 'longitude', 'precipitation_m', 'precipitation_mm'], dtype='str')


In [18]:
print("SSRD:")
print(ssrd["ssrd_kwh_m2"].describe())

SSRD:
count    7305.000000
mean        5.316464
std         1.509238
min         0.337581
25%         4.576471
50%         5.260571
75%         6.468090
max         7.997276
Name: ssrd_kwh_m2, dtype: float64


In [19]:
print("\nTemperature:")
print(temperature["temperature_c"].describe())


Temperature:
count    7305.000000
mean       27.003102
std         4.303237
min        13.586470
25%        23.941069
50%        27.388527
75%        29.967170
max        37.765283
Name: temperature_c, dtype: float64


In [20]:
print("\nPrecipitation:")
print(precipitation["precipitation_mm"].describe())


Precipitation:
count    7305.000000
mean        2.406637
std        11.513593
min        -0.000030
25%         0.000852
50%         0.001717
75%         0.382984
max       342.169568
Name: precipitation_mm, dtype: float64


In [21]:
climate = (
    ssrd[["date", "ssrd_kwh_m2"]]
    .merge(
        temperature[["date", "temperature_c"]],
        on="date",
        how="outer"
    )
    .merge(
        precipitation[["date", "precipitation_mm"]],
        on="date",
        how="outer"
    )
    .sort_values("date")
    .reset_index(drop=True)
)

display(climate.head())
display(climate.tail())

,date,ssrd_kwh_m2,temperature_c,precipitation_mm
0,2005-01-01,4.718580,17.744920,0.000840
1,2005-01-02,4.738148,18.059627,0.000866
2,2005-01-03,4.826562,18.268463,0.000858
3,2005-01-04,4.724119,19.491347,0.000855
4,2005-01-05,4.668332,19.993656,0.000861


,date,ssrd_kwh_m2,temperature_c,precipitation_mm
7300,2024-12-27,3.232900,22.026846,0.012237
7301,2024-12-28,4.549426,20.294033,0.000000
7302,2024-12-29,4.535698,20.296503,0.000858
7303,2024-12-30,4.537804,20.110393,0.000858
7304,2024-12-31,4.597704,20.340224,0.000000


In [22]:
print(climate.shape)
print(climate.isna().sum())

(7305, 4)
date                0
ssrd_kwh_m2         0
temperature_c       0
precipitation_mm    0
dtype: int64


In [23]:
master_file = CLIMATE_DIR / "ERA5_380006_2005_2024_master.csv"

climate.to_csv(
    master_file,
    index=False
)

print("Saved:", master_file)

Saved: c:\dev\case-study\raw\climate\ERA5_380006_2005_2024_master.csv
